# Design a binder to a real protein

**Protein design practical — University of Tartu**

You will choose a target protein, choose *where* on it to bind, choose what kind
of binder to make — and then let **BoltzGen** generate it, design its sequence,
fold the result and score it.

At the end you read the scorecard and decide whether you would actually make any
of them.

---

## How to use this notebook

1. **Runtime → Change runtime type → T4 GPU**, then Save.
2. Fill in your choices in the **one cell below**.
3. **Runtime → Run all**, then leave it. Setup takes ~10 minutes (about 6 GB of
   downloads) and the design run another ~10.

Everything after the choices cell runs by itself.


In [ ]:
#@title ## YOUR CHOICES — set these, then Runtime → Run all { display-mode: "form" }

#@markdown ### 1 · What do you want to bind?
target = "Ubiquitin (76 aa) - the classic signalling protein" #@param ["Protein G B1 domain (56 aa) - fastest", "Ubiquitin (76 aa) - the classic signalling protein", "Lysozyme (129 aa) - slowest"]

#@markdown ### 2 · Where on it?
#@markdown Real, named patches. Pick one you can justify — and pick one that
#@markdown belongs to the target you chose above.
binding_site = "Ile44 hydrophobic patch - what real ubiquitin-binding proteins use" #@param ["[Protein G] alpha-helix face - how it grips antibodies", "[Protein G] beta-strand edge - how it grips the other half of an antibody", "Ile44 hydrophobic patch - what real ubiquitin-binding proteins use", "[Ubiquitin] Phe4 patch - a second, less used surface", "[Ubiquitin] C-terminal tail - the end that attaches to other proteins", "[Lysozyme] active-site cleft - block the enzyme", "[Lysozyme] the D1.3 antibody epitope - the far side"]

#@markdown ### 3 · What kind of binder?
binder_type = "miniprotein - a small folded domain" #@param ["miniprotein - a small folded domain", "short peptide - linear, easy to synthesise", "cyclic peptide - rigid and protease-resistant"]
binder_length = 60 #@param {type:"slider", min:10, max:90, step:5}
#@markdown Typical lengths: miniprotein 45–75, short peptide 10–24, cyclic peptide 8–18.

#@markdown ### 4 · How hard to search?
num_designs = 4 #@param ["2", "4", "8"] {type:"raw"}
#@markdown Roughly 2.5 minutes per design.
run_name = "run1" #@param {type:"string"}

# ------------------------------------------------------------------ resolve
TARGETS = {
    "Protein G B1 domain (56 aa) - fastest":                   ("1PGA", "A"),
    "Ubiquitin (76 aa) - the classic signalling protein":      ("1UBQ", "A"),
    "Lysozyme (129 aa) - slowest":                             ("4LZT", "A"),
}
SITES = {
    "[Protein G] alpha-helix face - how it grips antibodies":  ("1PGA", 33),
    "[Protein G] beta-strand edge - how it grips the other half of an antibody": ("1PGA", 16),
    "Ile44 hydrophobic patch - what real ubiquitin-binding proteins use": ("1UBQ", 44),
    "[Ubiquitin] Phe4 patch - a second, less used surface":    ("1UBQ", 4),
    "[Ubiquitin] C-terminal tail - the end that attaches to other proteins": ("1UBQ", 72),
    "[Lysozyme] active-site cleft - block the enzyme":         ("4LZT", 35),
    "[Lysozyme] the D1.3 antibody epitope - the far side":                  ("4LZT", 116),
}
TYPES = {
    "miniprotein - a small folded domain":          ("protein-anything", False, 45, 75),
    "short peptide - linear, easy to synthesise":   ("peptide-anything", False, 10, 24),
    "cyclic peptide - rigid and protease-resistant": ("peptide-anything", True, 8, 18),
}

PDB_ID, CHAIN = TARGETS[target]
PROTOCOL, CYCLIC, LO_OK, HI_OK = TYPES[binder_type]
BUDGET = max(2, num_designs // 2)

if SITES[binding_site][0] != PDB_ID:
    ok = [k for k, v in SITES.items() if v[0] == PDB_ID]
    raise SystemExit("\nThat binding site is not on the target you chose.\n"
                     "Sites available on " + target.split(" (")[0] + ":\n  "
                     + "\n  ".join(ok))
HOTSPOT = SITES[binding_site][1]

if not (LO_OK <= binder_length <= HI_OK):
    print(f"NOTE: {binder_length} aa is unusual for this binder type "
          f"(typical {LO_OK}-{HI_OK}). Running anyway.")
LEN_MIN, LEN_MAX = max(6, binder_length - 8), binder_length + 8

print("=" * 64)
print(f"  target        {target.split(' (')[0]}   (PDB {PDB_ID}, chain {CHAIN})")
print(f"  binding site  residue {HOTSPOT}")
print(f"  binder        {binder_type.split(' -')[0]}, {LEN_MIN}-{LEN_MAX} aa"
      + ("  (cyclised)" if CYCLIC else ""))
print(f"  designs       {num_designs}, keeping best {BUDGET}")
print("=" * 64)
print("\nNow: Runtime -> Run all.  About 20 minutes start to finish.")


In [ ]:
#@title Setup - install BoltzGen and fetch weights  (~10 min)
%%time
import os, subprocess, sys

VENV = '/content/bgenv'
BG   = f'{VENV}/bin/boltzgen'
os.environ['HF_HOME'] = '/content/hf'

# Colab exports MPLBACKEND=module://matplotlib_inline.backend_inline, which exists
# only inside the Colab kernel. BoltzGen imports biotite -> matplotlib inside the
# 3.12 venv, where that backend is unknown and matplotlib raises on import.
BG_ENV = dict(os.environ, HF_HOME='/content/hf', MPLBACKEND='Agg')

def sh(cmd, **kw):
    print('$', cmd)
    kw.setdefault('env', BG_ENV)
    return subprocess.run(cmd, shell=True, check=True, **kw)

if not os.path.exists(BG):
    sh(f'{sys.executable} -m pip install -q uv')
    sh(f'{sys.executable} -m uv venv --python 3.12 {VENV}')
    sh(f'{sys.executable} -m uv pip install --python {VENV}/bin/python -q boltzgen')

sh(f'{sys.executable} -m pip install -q huggingface_hub py3Dmol pandas')

print('\nfetching model weights (~6 GB, once) ...')
from huggingface_hub import snapshot_download
snapshot_download('boltzgen/boltzgen-1', cache_dir='/content/hf/hub')
snapshot_download('boltzgen/inference-data', repo_type='dataset',
                  cache_dir='/content/hf/hub')

sh(f'{BG} --help > /dev/null')
print('\nBoltzGen ready.')


In [ ]:
#@title Fetch and clean your target
import urllib.request, os

raw = f'/content/{PDB_ID}.pdb'
if not os.path.exists(raw):
    urllib.request.urlretrieve(f'https://files.rcsb.org/download/{PDB_ID}.pdb', raw)

THREE2ONE = {'ALA':'A','ARG':'R','ASN':'N','ASP':'D','CYS':'C','GLN':'Q','GLU':'E',
             'GLY':'G','HIS':'H','ILE':'I','LEU':'L','LYS':'K','MET':'M','PHE':'F',
             'PRO':'P','SER':'S','THR':'T','TRP':'W','TYR':'Y','VAL':'V'}

clean, TARGET_SEQ, resids, seen, n = [], '', [], set(), 0
for l in open(raw):
    if not l.startswith('ATOM') or l[21] != CHAIN: continue
    if l[16] not in (' ', 'A'): continue
    if l[17:20].strip() not in THREE2ONE: continue
    n += 1
    clean.append(f'{l[:6]}{n:5d}{l[11:16]} {l[17:]}'.rstrip() + '\n')
    if l[22:27] not in seen:
        seen.add(l[22:27])
        TARGET_SEQ += THREE2ONE[l[17:20].strip()]
        resids.append(int(l[22:26]))

TARGET_PDB = '/content/target.pdb'
open(TARGET_PDB, 'w').writelines(clean + ['END\n'])

print(f'{PDB_ID} chain {CHAIN}: {len(resids)} residues, {n} atoms')
if HOTSPOT not in resids:
    raise SystemExit(f'Residue {HOTSPOT} is missing from this structure.')
pos = resids.index(HOTSPOT)
print(f'your site: {TARGET_SEQ[pos]}{HOTSPOT}\n')
for i in range(0, len(TARGET_SEQ), 60):
    row = TARGET_SEQ[i:i+60]
    mark = ''.join('^' if resids[i+j] == HOTSPOT else ' ' for j in range(len(row)))
    print(f'  {resids[i]:>4d}  {row}')
    if '^' in mark:
        print(f'        {mark}')


In [ ]:
#@title Run BoltzGen  (~10 min - this is the long one)
%%time
import subprocess, os, textwrap

OUT = f'/content/out_{run_name}'
cyc = '\n          cyclic: true' if CYCLIC else ''
yaml = textwrap.dedent(f"""\
    entities:
      - file:
          path: {TARGET_PDB}
          include:
            - chain:
                id: {CHAIN}
          binding_types:
           - chain:
               id: {CHAIN}
               binding: {HOTSPOT}

      - protein:
          id: C
          sequence: {LEN_MIN}..{LEN_MAX}{cyc}
    """)
open('/content/design.yaml', 'w').write(yaml)
print(yaml)

env = dict(os.environ, HF_HOME='/content/hf', MPLBACKEND='Agg')
cmd = (f'{BG} run /content/design.yaml --output {OUT} '
       f'--protocol {PROTOCOL} --cache /content/hf/hub '
       f'--inverse_fold_avoid "" '
       f'--num_designs {num_designs} --budget {BUDGET} --devices 1')
print('$', cmd, '\n')

proc = subprocess.run(cmd, shell=True, env=env, text=True,
                      stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
print(proc.stdout[-6000:])
if proc.returncode != 0:
    print('=' * 70)
    print(f'BoltzGen failed (exit {proc.returncode}). The error is in the lines above.')
    print('=' * 70)
else:
    print(f'\ndone -> {OUT}')


---
## The scorecard

BoltzGen writes ~200 columns per design. These decide whether one is worth making:

| Column | Meaning | Want |
|---|---|---|
| `design_to_target_iptm` | confidence in the interface | higher; >0.8 strong |
| `min_design_to_target_pae` | expected error across the interface, Å | lower; <5 promising |
| `bb_rmsd` | did the sequence refold to the designed shape | <2 Å |
| `liability_*` | chemical problems that make it unmakeable | zero |

**Read the liabilities before you get excited about the ipTM.**


In [ ]:
#@title Read the scorecard
import glob, os, subprocess, sys
try:
    import pandas as pd
except ModuleNotFoundError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'pandas'], check=True)
    import pandas as pd

BEST_SEQ = None
csv = f'{OUT}/final_ranked_designs/all_designs_metrics.csv'
if not os.path.exists(csv):
    print('No results - the design cell did not finish.')
else:
    df = pd.read_csv(csv).sort_values('design_to_target_iptm', ascending=False)
    cols = [c for c in ['id', 'design_to_target_iptm', 'min_design_to_target_pae',
                        'bb_rmsd', 'liability_num_violations',
                        'liability_violations_summary', 'designed_sequence']
            if c in df.columns]
    display(df[cols])

    best = df.iloc[0]
    BEST_SEQ = best['designed_sequence']
    iptm = best['design_to_target_iptm']
    print(f"\nBest by interface confidence: {best['id']}")
    print(f"  ipTM          {iptm:.3f}   "
          + ('strong' if iptm > 0.8 else 'ambiguous' if iptm > 0.6 else 'weak'))
    print(f"  interface pAE {best['min_design_to_target_pae']:.2f} A")
    v = best.get('liability_violations_summary')
    if isinstance(v, str) and v.strip():
        print(f"  liabilities: {v}")
        print('     -> chemical problems regardless of the binding score')
    else:
        print('  no developability liabilities flagged')
    print(f"\n  sequence ({len(BEST_SEQ)} aa): {BEST_SEQ}")


---
## Confidence — pLDDT and pAE

**pLDDT** is per-residue. BoltzGen puts it in the B-factor column of the structure
on a **0–1 scale** (0.8 here means pLDDT 80). Colouring by it shows *which part*
of your binder the model is unsure about — usually termini and floppy loops.

**pAE** is pairwise. BoltzGen does not write the full matrix, but gives the number
that matters for a binder: expected error *across the interface*.

| | good | marginal | poor |
|---|---|---|---|
| pLDDT (0–1) | > 0.8 | 0.6–0.8 | < 0.6 |
| interface pAE (Å) | < 5 | 5–10 | > 10 |


In [ ]:
#@title Colour by pLDDT, read the interface pAE
import glob, os, collections
import numpy as np, matplotlib.pyplot as plt
try:
    import py3Dmol
except ModuleNotFoundError:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'py3Dmol'], check=True)
    import py3Dmol

f = sorted(glob.glob(f'{OUT}/final_ranked_designs/final_*_designs/*.cif'))
if not f:
    print('No structures found.')
else:
    path = f[0]
    hdr, rows, inloop = [], [], False
    for ln in open(path):
        if ln.startswith('_atom_site.'):
            hdr.append(ln.strip().split('.')[1]); inloop = True; continue
        if inloop and ln.startswith(('ATOM', 'HETATM')): rows.append(ln.split())
        elif inloop and ln.startswith('#') and rows: break
    ci, si = hdr.index('label_asym_id'), hdr.index('label_seq_id')
    bi, ai = hdr.index('B_iso_or_equiv'), hdr.index('label_atom_id')

    per_chain = collections.defaultdict(list)
    for r in rows:
        try: per_chain[r[ci]].append(float(r[bi]))
        except ValueError: pass
    DESIGN_CHAIN = min(per_chain, key=lambda c: len(per_chain[c]))

    print('mean pLDDT by chain:')
    for ch, v in per_chain.items():
        label = 'your binder' if ch == DESIGN_CHAIN else 'target'
        print(f'   chain {ch} ({label:11s}) {np.mean(v):.3f}')

    ca = sorted((int(r[si]), float(r[bi])) for r in rows
                if r[ci] == DESIGN_CHAIN and r[ai].strip(chr(34)) == 'CA')
    resi = [i for i, _ in ca]
    vals = np.array([v for _, v in ca])

    plt.figure(figsize=(9, 3))
    plt.plot(resi, vals, '-o', ms=3)
    plt.axhspan(.8, 1., color='tab:blue', alpha=.12, label='confident')
    plt.axhspan(.6, .8, color='tab:olive', alpha=.12, label='uncertain')
    plt.axhspan(0., .6, color='tab:orange', alpha=.12, label='low')
    plt.ylim(0, 1)
    plt.xlabel(f'residue in your binder (chain {DESIGN_CHAIN})')
    plt.ylabel('pLDDT'); plt.legend(fontsize=8); plt.grid(alpha=.3)
    plt.title(f'per-residue confidence - mean {vals.mean():.3f}')
    plt.tight_layout(); plt.show()

    view = py3Dmol.view(width=760, height=560)
    view.addModel(open(path).read(), 'cif')
    view.setStyle({'cartoon': {'colorscheme':
                   {'prop': 'b', 'gradient': 'roygb', 'min': .4, 'max': .9}}})
    view.setBackgroundColor('white'); view.zoomTo(); view.show()
    print('\nblue = confident, green/yellow = middling, red = the model is guessing')
    print(f'\nAlso written: {OUT}/final_ranked_designs/results_overview.pdf '
          f'(32 pages of distributions)')


In [ ]:
#@title Did it bind where you asked?
import glob, numpy as np, py3Dmol

f = sorted(glob.glob(f'{OUT}/final_ranked_designs/final_*_designs/*.cif'))
if f:
    path = f[0]
    hdr, rows, inloop = [], [], False
    for ln in open(path):
        if ln.startswith('_atom_site.'):
            hdr.append(ln.strip().split('.')[1]); inloop = True; continue
        if inloop and ln.startswith(('ATOM', 'HETATM')): rows.append(ln.split())
        elif inloop and ln.startswith('#') and rows: break
    ci, si, xi = (hdr.index('label_asym_id'), hdr.index('label_seq_id'),
                  hdr.index('Cartn_x'))
    tgt = [r for r in rows if r[ci] == CHAIN]
    bnd = [r for r in rows if r[ci] != CHAIN]
    T = np.array([[float(r[xi]), float(r[xi+1]), float(r[xi+2])] for r in tgt])
    B = np.array([[float(r[xi]), float(r[xi+1]), float(r[xi+2])] for r in bnd])
    d = np.linalg.norm(B[:, None, :] - T[None, :, :], axis=-1)
    touched = sorted({int(tgt[j][si]) for j in np.where((d < 5.0).any(axis=0))[0]})
    touched_pdb = sorted({resids[i-1] for i in touched if 0 < i <= len(resids)})
    near = [r for r in touched_pdb if abs(r - HOTSPOT) <= 5]

    print(f'binder contacts {len(touched_pdb)} target residues: {touched_pdb}')
    print(f'you asked for residue {HOTSPOT}')
    if HOTSPOT in touched_pdb:
        print('  it bound your site directly')
    elif near:
        print(f'  it landed nearby ({near}) but not on your residue')
    else:
        print('  it ignored your site and bound elsewhere.')
        print('  The hotspot is a preference, not a constraint. This happens.')

    binder_chain = next(c for c in {r[ci] for r in rows} if c != CHAIN)
    view = py3Dmol.view(width=760, height=560)
    view.addModel(open(path).read(), 'cif')
    view.setStyle({'chain': CHAIN}, {'cartoon': {'color': 'lightgrey'}})
    view.setStyle({'chain': binder_chain}, {'cartoon': {'color': 'orange'}})
    view.addStyle({'chain': CHAIN, 'resi': [str(HOTSPOT)]},
                  {'stick': {'colorscheme': 'redCarbon'}})
    view.setBackgroundColor('white'); view.zoomTo(); view.show()
    print('\ngrey = target, orange = your binder, red sticks = the site you chose')


---
## Check it with an independent model

BoltzGen folded your design with its *own* machinery, so of course it agrees with
itself. A genuinely independent predictor is a far stronger test — and
[AlphaFold Server](https://alphafoldserver.com) is free and takes about a minute.


In [ ]:
#@title Sequences for AlphaFold Server
print('Go to  https://alphafoldserver.com   (sign in with a Google account)')
print('Add TWO protein entities and paste one sequence into each.\n')
print('=' * 64)
print(f'ENTITY 1 - the target ({PDB_ID} chain {CHAIN}, {len(TARGET_SEQ)} aa)')
print('=' * 64)
print(TARGET_SEQ)
if BEST_SEQ:
    print()
    print('=' * 64)
    print(f'ENTITY 2 - your binder ({len(BEST_SEQ)} aa)')
    print('=' * 64)
    print(BEST_SEQ)
else:
    print('\n(run the scorecard cell first to get your binder sequence)')
print("""
Then Continue, then Submit.

What to look for:
  * does AlphaFold put your binder in the same place BoltzGen did?
  * the PAE plot - a dark off-diagonal block between the two chains means
    AlphaFold is confident where they sit relative to each other. A pale
    block means it has no idea, and the two chains are independent.
  * ipTM on the results page, same scale as BoltzGen's.

If AlphaFold disagrees with BoltzGen, that disagreement is the most useful
result you will get today.
""")


In [ ]:
#@title Download everything
import subprocess, os
from google.colab import files
if os.path.isdir(OUT):
    subprocess.run(f'zip -qr /content/{run_name}.zip {OUT}', shell=True)
    files.download(f'/content/{run_name}.zip')
else:
    print('Nothing to download - the design run did not finish.')


---
## If you have time

- **Run again with a different site** on the same target. Does the binder follow
  you, or go back where it went the first time?
- **Change the binder type.** A cyclic peptide is a very different molecule from
  a miniprotein, and the scores will show it.
- **Sort the scorecard by `liability_num_violations`** instead of ipTM. Does the
  ranking change? Which ordering would you trust?

## For the discussion

Bring your best design's **ipTM**, its **liabilities**, whether it bound where you
asked, and whether AlphaFold agreed. Then say whether you would spend money
making it.

---

Published campaigns generate **10,000–60,000** designs and test ~150. You ran a
handful. Keep that in proportion.

BoltzGen: Stark et al., 2025 (MIT).
